# Noise on real hardware

| | |
|---|---|
| **Level** | Introductory to intermediate |
| **Time** | About 45 to 60 minutes |
| **Prerequisites** | CNOT, measurement. Expectation values are helpful but not required. |
| **Default device** | Rigetti Cepheus-1-108Q |
| **Also runs on** | IQM Garnet |
| **Qubits** | 4 |
| **Two-qubit gates** | 3 per random layer, up to 24 at the default setting |
| **Hardware jobs** | 4 (one per depth) |
| **Approximate cost** | Rigetti: about 40 credits in total (billed by execution time). Garnet at 500 shots: about 410 credits. |
| **Suggested hand-in** | Your fidelity plot with the hardware points, and answers to Questions 1 to 3 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

Adapted from notebook 1 of the qBraid [Error-Mitigation series](https://github.com/qBraid/Error-Mitigation) (`tutorials/Error-Mitigation/` in this repository).

*Part of the QUEST notebooks from qBraid. You may copy, edit and adapt this notebook for your course.*

Every result from today's quantum hardware contains errors. This notebook introduces the three main kinds, shows that each one leaves a different pattern in the results, and then runs an experiment on a real device to see which pattern it shows.

Unlike the other starter notebooks, this one also uses **noisy simulation**: the simulator adds errors from a model we specify. Here noise is the subject, so we build it up one kind at a time.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "rigetti:rigetti:qpu:cepheus-1-108q"   # device list and prices: see the README
SHOTS = 1000                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits
HW_DEPTHS = [1, 2, 4, 8]   # random layers to run on hardware, one job each
QUEST_JOB_TAGS = {"quest": "noise-depth"}   # labels this notebook's hardware jobs for QUEST usage statistics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error, thermal_relaxation_error
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

import re

_NOT_GATES = {"openqasm", "include", "bit", "qubit", "box", "measure", "declare", "pragma",
              "b", "c", "meas", "ro", "barrier", "fence", "delay", "halt", "reset", "defcal", "cal"}

def _count_gates(program_text):
    """(all gates, two-qubit gates) in a compiled OpenQASM or Quil program."""
    total = two = 0
    for line in program_text.splitlines():
        m = re.match(r"\s*([A-Za-z_]+)", line)
        if not m or m.group(1).lower() in _NOT_GATES:
            continue
        total += 1
        if m.group(1).lower() in ("cz", "cx", "cnot", "iswap", "xy", "cphase", "ecr", "ms", "zz"):
            two += 1
    return total, two

def check_what_ran(job, circuit):
    """Compare the circuit we sent with the program the device actually ran.

    Device compilers rewrite circuits before running them. Usually that only
    changes the gate names, but a compiler can also remove gates that cancel,
    such as a circuit followed by its inverse. This prints both gate counts.
    """
    ops = [inst.operation for inst in circuit.data if inst.operation.name not in ("barrier", "measure")]
    sent_total, sent_two = len(ops), sum(1 for op in ops if op.num_qubits == 2)
    try:
        program = job.client.get_job_compiled_program(job.id)
    except Exception as err:
        print(f"  could not fetch the compiled program ({type(err).__name__}); check skipped")
        return None
    ran_total, ran_two = _count_gates(getattr(program, "data", str(program)))
    print(f"  gates sent {sent_total} ({sent_two} two-qubit); device ran {ran_total} ({ran_two} two-qubit)")
    removed = (sent_two and ran_two < 0.5 * sent_two) or (sent_total >= 10 and ran_total < 0.5 * sent_total)
    if removed:
        print("  WARNING: the compiler removed most of the gates. "
              "This result does not measure the circuit you built.")
    return not removed

## 1. The experiment: a mirror circuit

We need a circuit whose correct answer we know exactly, so that any deviation must be noise. A **mirror circuit** does this. It applies a block of gates $U$, then its inverse $U^\dagger$. The two cancel, so the qubits should end exactly where they started, in $|0000\rangle$.

Repeating the pair $k$ times makes the circuit deeper without changing the answer. That lets us see how errors grow with circuit length.

We summarise each result with the parity $\langle ZZZZ\rangle$: +1 for each shot with an even number of 1s, and -1 for each shot with an odd number. The ideal value is exactly +1.

In [ ]:
N_QUBITS = 4
ANGLES = [0.63, -1.18, 0.41, 0.97]

def mirror_half():
    """U: a layer of Ry rotations followed by a ladder of CNOTs."""
    qc = QuantumCircuit(N_QUBITS)
    for q in range(N_QUBITS):
        qc.ry(ANGLES[q], q)
    for q in range(N_QUBITS - 1):
        qc.cx(q, q + 1)
    return qc

def mirror_circuit(layers, measure=True):
    """(U then U-dagger), repeated `layers` times. Logically the identity."""
    u = mirror_half()
    qc = QuantumCircuit(N_QUBITS)
    for _ in range(layers):
        qc.compose(u, inplace=True)
        qc.barrier()                 # keeps Qiskit from merging U with U-dagger
        qc.compose(u.inverse(), inplace=True)
        qc.barrier()
    if measure:
        qc.measure_all()
    return qc

def zzzz(counts):
    """Parity <ZZZZ>: +1 for even numbers of 1s, -1 for odd."""
    total = sum(counts.values())
    return sum((-1) ** key.replace(" ", "").count("1") * n for key, n in counts.items()) / total

mirror_circuit(1).draw(output="text", fold=120)

## 2. Ideal simulation

First we check that the mirror really is the identity, then that a perfect simulator returns +1.

In [ ]:
assert np.allclose(Operator(mirror_circuit(3, measure=False)).data, np.eye(16), atol=1e-10)

def sample(circuit, noise_model=None, shots=8192):
    simulator = AerSimulator(noise_model=noise_model)
    result = simulator.run(circuit, shots=shots, seed_simulator=7).result()
    return result.get_counts()

ideal = sample(mirror_circuit(1))
print("ideal counts:", ideal)
print(f"ideal <ZZZZ>: {zzzz(ideal):+.3f}")

## 3. Three kinds of noise

Almost all errors on real devices fall into three categories. We build a noise model for each one separately. The numbers are typical of current superconducting devices; real devices vary from qubit to qubit and from day to day.

- **Readout error.** The qubit is in the right state, but the measurement reports the wrong bit. We set $P(1|0) = 1\%$ and $P(0|1) = 3\%$. Mistakes from 1 to 0 are usually more common, because a qubit in $|1\rangle$ can relax during the measurement.
- **Gate error.** Each gate is slightly wrong. We use the standard depolarizing model: after a gate, with a small probability, the qubit's state is replaced by a random one. We use $10^{-3}$ for one-qubit gates and $10^{-2}$ for two-qubit gates.
- **Decoherence.** Even an idle qubit decays. $T_1$ is the time for $|1\rangle$ to relax to $|0\rangle$. $T_2$ is the time for a superposition to lose its phase. The damage per gate depends on how long the gate takes. We use $T_1 = 50\,\mu s$, $T_2 = 30\,\mu s$, 50 ns one-qubit gates and 200 ns two-qubit gates, as illustrative values.

In [ ]:
# 1. Readout error only
readout_model = NoiseModel()
readout_model.add_all_qubit_readout_error(ReadoutError([[0.99, 0.01], [0.03, 0.97]]))

# 2. Gate error only (depolarizing)
gate_model = NoiseModel()
gate_model.add_all_qubit_quantum_error(depolarizing_error(1e-3, 1), ["ry", "rz"])
gate_model.add_all_qubit_quantum_error(depolarizing_error(1e-2, 2), ["cx"])

# 3. Decoherence only (T1 and T2 during each gate). Times in nanoseconds.
T1, T2, T_1Q, T_2Q = 50_000, 30_000, 50, 200
thermal_model = NoiseModel()
thermal_model.add_all_qubit_quantum_error(thermal_relaxation_error(T1, T2, T_1Q), ["ry", "rz"])
thermal_model.add_all_qubit_quantum_error(
    thermal_relaxation_error(T1, T2, T_2Q).tensor(thermal_relaxation_error(T1, T2, T_2Q)), ["cx"])

# All three together
full_model = NoiseModel()
full_model.add_all_qubit_readout_error(ReadoutError([[0.99, 0.01], [0.03, 0.97]]))
full_model.add_all_qubit_quantum_error(
    depolarizing_error(1e-3, 1).compose(thermal_relaxation_error(T1, T2, T_1Q)), ["ry", "rz"])
full_model.add_all_qubit_quantum_error(
    depolarizing_error(1e-2, 2).compose(
        thermal_relaxation_error(T1, T2, T_2Q).tensor(thermal_relaxation_error(T1, T2, T_2Q))), ["cx"])

models = {"readout only": readout_model, "gate only": gate_model,
          "decoherence only": thermal_model, "all three": full_model}
for name, model in models.items():
    print(f"{name:17s} <ZZZZ> at 2 layers: {zzzz(sample(mirror_circuit(2), model)):+.3f}")

## 4. Each kind of noise leaves a different pattern

The correct answer is `0000`. The plots show where the wrong shots go, on a log scale so the small counts are visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, name in zip(axes, ["readout only", "gate only", "decoherence only"]):
    counts = sample(mirror_circuit(2), models[name])
    total = sum(counts.values())
    top = sorted(counts, key=counts.get, reverse=True)[:8]
    ax.bar(top, [counts[k] / total for k in top],
           color=["tab:blue" if k == "0000" else "tab:red" for k in top])
    ax.set_yscale("log")
    ax.set_title(name, fontsize=10)
    ax.tick_params(axis="x", rotation=75)
axes[0].set_ylabel("fraction of shots (log scale)")
plt.show()

- **Readout error** sends shots almost only to outcomes with a single wrong bit (`0001`, `0010`, `0100`, `1000`). Each qubit's measurement fails independently, and two failures at once are rare.
- **Gate error** spreads shots across many outcomes, including several wrong bits at once. An error partway through the circuit is spread to other qubits by the CNOTs that follow.
- **Decoherence** looks mild here, because the correct answer `0000` is also the state that $T_1$ relaxes towards. On a circuit whose answer is not all zeros, it pulls results towards 0.

## 5. The depth test

The pattern that best separates the three kinds of noise is how the result changes with circuit depth.

In [ ]:
depths = np.arange(1, 9)
decay = {name: [zzzz(sample(mirror_circuit(int(k)), model)) for k in depths]
         for name, model in models.items()}

def plot_signatures():
    colors = {"readout only": "tab:cyan", "gate only": "tab:blue",
              "decoherence only": "tab:olive", "all three": "tab:red"}
    plt.plot(depths, np.ones_like(depths), "--", color="gray", label="ideal")
    for name, values in decay.items():
        plt.plot(depths, values, "o-", color=colors[name], label=name)
    plt.xlabel("mirror layers (depth grows, ideal answer stays +1)")
    plt.ylabel("<ZZZZ>")
    plt.ylim(0, 1.08)

plot_signatures()
plt.legend(loc="lower left")
plt.show()

- **Readout error is flat.** It affects only the final measurement, so it does not depend on depth.
- **Gate error falls off exponentially.** Each layer adds more gates, and each gate adds a little error.
- **Decoherence also falls,** more slowly here, at a rate set by the circuit's duration compared with $T_1$ and $T_2$.
- **A real device has all three at once,** which is the red curve.

## 6. Why the hardware run uses a different circuit

Before a device runs a circuit, its compiler rewrites it into the device's native gates. Compilers also remove gates that have no overall effect, and a mirror circuit is exactly that: $U$ followed by $U^\dagger$ does nothing. In our tests, device compilers removed most or all of the mirror's gates, barriers included, so the device measured little more than readout error.

So on hardware we use a circuit that cannot be simplified: layers of random rotations, each followed by a ladder of CNOTs. Its correct answer is not a single bitstring but a spread of outcomes, some likely and some unlikely, which the ideal simulator computes for us. We score a result by how often the device lands on the outcomes the ideal calculation says are likely, compared with random guessing. The score is scaled so that a perfect device scores 1 and a device returning random bits scores 0. It is the **cross-entropy fidelity** used to benchmark quantum processors, including in Google's 2019 quantum-supremacy experiment.

With a finite number of shots the score is only an estimate: at 1000 shots, even a perfect device scores within about 0.1 of 1.

After each hardware job, `check_what_ran` compares the number of two-qubit gates we sent with the number the device actually ran, so you can confirm that your circuit reached the device.

In [ ]:
LAYER_SEED = 2026

def random_layers(layers, measure=True):
    """`layers` layers of random rotations, each followed by a CNOT ladder.
    The same seed gives the same first layers at every depth."""
    rng = np.random.default_rng(LAYER_SEED)
    qc = QuantumCircuit(N_QUBITS)
    for _ in range(layers):
        for q in range(N_QUBITS):
            qc.ry(rng.uniform(0.3, 2.8), q)
            qc.rz(rng.uniform(0.3, 2.8), q)
        for q in range(N_QUBITS - 1):
            qc.cx(q, q + 1)
    if measure:
        qc.measure_all()
    return qc

def ideal_probs(layers):
    """Exact outcome probabilities of random_layers(layers), from the state vector."""
    return Statevector(random_layers(layers, measure=False)).probabilities_dict()

def fidelity_score(counts, probs):
    """Linear cross-entropy fidelity: 1 for a perfect device, 0 for random bits."""
    total = sum(counts.values())
    measured = {k.replace(" ", ""): n / total for k, n in counts.items()}
    uniform = 1 / 2 ** N_QUBITS
    hits = sum(measured.get(k, 0.0) * p for k, p in probs.items())   # average ideal probability of what we measured
    perfect = sum(p * p for p in probs.values())                      # the same average for a perfect device
    return (hits - uniform) / (perfect - uniform)

layer_counts = np.array([1, 2, 4, 6, 8, 12, 16])
rand_decay = {name: [fidelity_score(sample(random_layers(int(k)), model), ideal_probs(int(k)))
                     for k in layer_counts]
              for name, model in models.items()}

def plot_scores():
    colors = {"readout only": "tab:cyan", "gate only": "tab:blue",
              "decoherence only": "tab:olive", "all three": "tab:red"}
    plt.plot(layer_counts, np.ones_like(layer_counts), "--", color="gray", label="ideal")
    plt.plot(layer_counts, np.zeros_like(layer_counts), ":", color="gray", label="random bits")
    for name, values in rand_decay.items():
        plt.plot(layer_counts, values, "o-", color=colors[name], label=name)
    plt.xlabel("random layers (3 CNOTs each)")
    plt.ylabel("fidelity score")
    plt.ylim(-0.1, 1.15)

random_layers(1).draw(output="text", fold=120)

In [ ]:
plot_scores()
plt.legend(loc="lower left", fontsize=8)
plt.show()

The same patterns appear as in section 5. Readout error lowers the score by a similar amount at every depth. Gate error and decoherence pull it further down with every layer, towards the random-bits line at 0.

## 7. Run the depth test on hardware

The next cell estimates the cost of running the depths listed in `HW_DEPTHS`, one job per depth.

In [ ]:
N_JOBS = len(HW_DEPTHS)

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    hw_circuits = {k: random_layers(k) for k in HW_DEPTHS}
    jobs = {k: device.run(c, shots=SHOTS, tags=QUEST_JOB_TAGS) for k, c in hw_circuits.items()}
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = {}
    for k, job in jobs.items():
        hw_counts[k] = job.result().data.get_counts()
        print(f"{k} layers: fidelity score = {fidelity_score(hw_counts[k], ideal_probs(k)):+.3f}")
        check_what_ran(job, hw_circuits[k])
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 8. Compare

In [ ]:
plot_scores()
if hw_counts:
    plt.plot(HW_DEPTHS, [fidelity_score(hw_counts[k], ideal_probs(k)) for k in HW_DEPTHS], "s",
             color="black", markersize=9, label=f"hardware ({DEVICE_ID})")
plt.legend(loc="lower left", fontsize=8)
plt.show()

Compare the hardware points with the simulated curves.

Look for where the hardware score starts to fall and how fast it approaches the random-bits line. Devices, and qubits on the same device, differ, so the curve you measure is specific to the device and the day. At 100 shots each point is uncertain by about 0.15; the default 1000 shots gives smoother curves.

The noise models above are simpler than a real device. They leave out crosstalk between neighbouring qubits, errors that add up coherently from gate to gate, and the fact that some qubits and connections are much worse than the median.

## Questions to try

1. Which simulated curve does the hardware resemble most?
2. At 1 layer the hardware score is already below 1. Which kind of noise causes a loss that is the same at every depth?
3. Did `check_what_ran` report the same number of two-qubit gates sent and run? If the device ran fewer, what does that mean for your result?
4. Run the same experiment on IQM Garnet (change `DEVICE_ID`). Which device stays closer to the ideal?
5. Change the gate error in the noise model until the simulated "all three" curve matches your hardware points. What two-qubit error rate do you need?
6. At what depth does the hardware reach the random-bits line? Add depths to `HW_DEPTHS` to find out; each one is another hardware job.
7. Optional, one extra job: run `mirror_circuit(4)` on hardware and call `check_what_ran` on the job. What did the device actually run?

## Going further

The qBraid Error-Mitigation series continues from here. Its notebook 2 corrects readout error, notebook 3 introduces zero-noise extrapolation, and notebook 4 combines them on real hardware. Zero-noise extrapolation deliberately adds gates that cancel, so check what the device ran before trusting it.